In [1]:
import torch
import torch.nn as nn
import time
import numpy as np

torch.set_num_threads(4)  # control CPU threads


Simple CPU model (MLP)

In [2]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, 10),
        )

    def forward(self, x):
        return self.net(x)

model = MLP().eval()


Warm-up (IMPORTANT

In [3]:
for _ in range(10):
    x = torch.randn(32, 512)
    model(x)


Benchmark latency (batch size = 1 vs 32)

In [5]:
def benchmark(batch_size, runs=100):
    times = []
    x = torch.randn(batch_size, 512)

    for _ in range(runs):
        start = time.perf_counter()
        model(x)
        end = time.perf_counter()
        times.append(end - start)

    times = np.array(times)
    return {
        "batch": batch_size,
        "mean_ms": times.mean() * 1000,
        "p95_ms": np.percentile(times, 95) * 1000,
        "throughput": batch_size / times.mean()
    }

print(benchmark(1))
print(benchmark(32))


{'batch': 1, 'mean_ms': 0.7283150000002081, 'p95_ms': 1.2751700000066535, 'throughput': 1373.0322731231875}
{'batch': 32, 'mean_ms': 1.4165190000008465, 'p95_ms': 2.015439999994584, 'throughput': 22590.590030900312}


PyTorch CPU Profiler

In [6]:
with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU],
    record_shapes=True
) as prof:
    x = torch.randn(32, 512)
    model(x)

print(prof.key_averages().table(
    sort_by="cpu_time_total",
    row_limit=10
))


----------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                  Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
----------------------  ------------  ------------  ------------  ------------  ------------  ------------  
          aten::linear         4.64%     115.000us        67.49%       1.673ms     557.667us             3  
           aten::addmm        48.45%       1.201ms        50.63%       1.255ms     418.333us             3  
           aten::randn        20.61%     511.000us        28.08%     696.000us     696.000us             1  
               aten::t         6.66%     165.000us        12.22%     303.000us     101.000us             3  
       aten::transpose         5.32%     132.000us         5.57%     138.000us      46.000us             3  
           aten::empty         5.08%     126.000us         5.08%     126.000us     126.000us             1  
            aten::r

In [7]:
for t in [1, 2, 4]:
    torch.set_num_threads(t)
    print(f"Threads={t}", benchmark(32))


Threads=1 {'batch': 32, 'mean_ms': 1.5771110000008548, 'p95_ms': 3.3201699999864327, 'throughput': 20290.264921101087}
Threads=2 {'batch': 32, 'mean_ms': 1.167939000000331, 'p95_ms': 2.025590000006616, 'throughput': 27398.691198762037}
Threads=4 {'batch': 32, 'mean_ms': 1.8602470000007543, 'p95_ms': 3.0162450000076984, 'throughput': 17202.016721428405}


Simulate slow preprocessing

In [10]:
def slow_preprocessing(x):
    out = []
    for row in x:
        out.append(row.numpy().mean())  # BAD: Python loop
    return torch.tensor(out)

x = torch.randn(32, 512)


In [8]:
for t in [1, 2, 4]:
    torch.set_num_threads(t)
    print(f"Threads={t}", benchmark(32))


Threads=1 {'batch': 32, 'mean_ms': 2.3453030000021613, 'p95_ms': 4.435449999984086, 'throughput': 13644.29244322397}
Threads=2 {'batch': 32, 'mean_ms': 1.4824780000000715, 'p95_ms': 2.716809999975567, 'throughput': 21585.48052652279}
Threads=4 {'batch': 32, 'mean_ms': 1.2257099999990828, 'p95_ms': 1.8183849999928723, 'throughput': 26107.317391572185}


In [11]:
start = time.perf_counter()
slow_preprocessing(x)
print("Preprocessing ms:", (time.perf_counter() - start) * 1000)

start = time.perf_counter()
model(x)
print("Model ms:", (time.perf_counter() - start) * 1000)


Preprocessing ms: 7.684900000015205
Model ms: 1.8278000000009342


In [12]:
import cProfile
import pstats

profiler = cProfile.Profile()
profiler.enable()

slow_preprocessing(x)
model(x)

profiler.disable()

stats = pstats.Stats(profiler)
stats.sort_stats("cumtime").print_stats(10)


         502 function calls (490 primitive calls) in 0.003 seconds

   Ordered by: cumulative time
   List reduced from 51 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        3    0.000    0.000    0.003    0.001 c:\Users\jredo\anaconda3\envs\onnx-test\lib\site-packages\IPython\core\interactiveshell.py:3514(run_code)
        3    0.000    0.000    0.003    0.001 {built-in method builtins.exec}
        1    0.000    0.000    0.002    0.002 C:\Users\jredo\AppData\Local\Temp\ipykernel_14820\2445718420.py:8(<module>)
      7/1    0.000    0.000    0.002    0.002 c:\Users\jredo\anaconda3\envs\onnx-test\lib\site-packages\torch\nn\modules\module.py:1514(_wrapped_call_impl)
      7/1    0.000    0.000    0.002    0.002 c:\Users\jredo\anaconda3\envs\onnx-test\lib\site-packages\torch\nn\modules\module.py:1520(_call_impl)
        1    0.000    0.000    0.002    0.002 C:\Users\jredo\AppData\Local\Temp\ipykernel_14820\3987173608.py:12(forwa

In [13]:
def fast_preprocessing(x):
    return x.mean(dim=1)

start = time.perf_counter()
fast_preprocessing(x)
print("Fast preprocessing ms:", (time.perf_counter() - start) * 1000)


Fast preprocessing ms: 15.45490000000882


In [14]:
x = torch.randn(32, 512)

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU],
    record_shapes=True
) as prof:
    model(x)

print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))


----------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                  Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
----------------------  ------------  ------------  ------------  ------------  ------------  ------------  
          aten::linear         1.42%      25.000us        92.38%       1.625ms     541.667us             3  
           aten::addmm        81.64%       1.436ms        86.07%       1.514ms     504.667us             3  
            aten::relu         1.93%      34.000us         7.62%     134.000us      67.000us             2  
       aten::clamp_min         5.69%     100.000us         5.69%     100.000us      50.000us             2  
               aten::t         3.07%      54.000us         4.89%      86.000us      28.667us             3  
           aten::copy_         3.87%      68.000us         3.87%      68.000us      22.667us             3  
       aten::transp